Version control: 2026-10-14 - 18:30
Luis Anunciacao, PhD

# DA - AGEM Time Series

Loads `df.rds` from the DP notebook. Nothing is transformed here.

**Tests.** Items are sorted by their smallest p. Two-sided Wilcoxon rank-sum (Mann-Whitney U) tests between consecutive waves, by company, for scale items. Waves are independent samples: SurveyMonkey collected no respondent ID, so answers cannot be paired. Lideres T2 items 6 and 20 are reverse-coded for the tests only (DP §2); counts are raw. "—" = item not applied in that wave. Tables 3-4 repeat Tables 1-2 with the scale responses collapsed to 3 categories (discordo / neutro / concordo; nunca/raramente / as vezes / muitas vezes/sempre); the tests are the same (5-point). Tables 3-4 flag risk items (negatively worded) "(risco)" (DP §2) and drop the questionnaire's "(invertida)" tags.

In [1]:
pacman::p_load(tidyverse)

options(repr.matrix.max.rows = 400, repr.matrix.max.cols = 20)

# DP output from GitHub (Processing and analysis/Importable data/df.rds); private repository, token from the git credential store (gitcreds)
df <- readRDS(gzcon(curl::curl("https://raw.githubusercontent.com/anovabr/AGEM-Time-Series/main/Processing%20and%20analysis/Importable%20data/df.rds",
                               handle = curl::new_handle() |> curl::handle_setheaders(Authorization = paste("token", gitcreds::gitcreds_get()$password)))))

# item x response frequency table by company and wave, with W and p of the rank-sum test (T1 vs T2, T2 vs T3) on the first row of each item;
# items sorted by their smallest p. agrupar = TRUE: counts of the 3-category responses (codigo_3 / resposta_3), risk items (negatively worded) flagged
# "(risco)" (the questionnaire's own "(invertida)" tags on positive items are dropped); the tests always use the 5-point valor_teste
tabela <- function(inst, agrupar = FALSE) {
  d <- filter(df, instrumento == inst) |> mutate(codigo = if (agrupar) codigo_3 else codigo)
  o <- filter(attr(df, "opcoes"), instrumento == inst) |> transmute(item, codigo = if (agrupar) codigo_3 else codigo, resposta = if (agrupar) resposta_3 else resposta) |> distinct()
  o |>
    inner_join(distinct(d, item, empresa, onda), by = "item", relationship = "many-to-many") |>
    left_join(count(d, item, codigo, empresa, onda), by = c("item", "codigo", "empresa", "onda")) |>
    mutate(n = as.character(replace_na(n, 0L))) |>
    bind_rows(d |>
                filter(!is.na(valor_teste)) |>
                reframe(map_dfr(list(c("T1", "T2"), c("T2", "T3")), \(w) if (all(w %in% onda))
                          with(suppressWarnings(wilcox.test(valor_teste[onda == w[1]], valor_teste[onda == w[2]])),
                               tibble(par = paste0(w[1], "→", w[2]), p = p.value,
                                      n = paste0(sprintf("W = %.1f; ", statistic), if_else(p.value < 0.001, "p < 0.001", sprintf("p = %.3f", p.value)))))),
                        .by = c(item, empresa)) |>
                transmute(item, empresa, onda = par, n, p, codigo = 1L)) |>
    mutate(ordem = suppressWarnings(min(p, na.rm = TRUE)), .by = item) |>
    pivot_wider(id_cols = c(item, codigo, ordem), names_from = c(empresa, onda), values_from = n, names_sort = TRUE) |>
    arrange(ordem, item, codigo) |>
    left_join(distinct(d, item, item_texto, risco), by = "item") |>
    left_join(select(o, item, codigo, resposta), by = c("item", "codigo")) |>
    mutate(item_texto = if (agrupar) paste0(str_remove(item_texto, " \\(invertida\\)"), if_else(risco, " (risco)", "")) else item_texto,
           Item = if_else(codigo == 1, paste0(item, ". ", item_texto), ""), Resposta = paste0("(", codigo, ") ", resposta),
           across(-c(item, codigo, ordem, item_texto, risco, resposta, Item, Resposta), ~ if_else(is.na(.) & (!str_detect(cur_column(), "→") | codigo == 1), "—", replace_na(., "")))) |>
    (\(t) select(t, Item, Resposta, all_of(sort(setdiff(names(t), c("item", "codigo", "ordem", "item_texto", "risco", "resposta", "Item", "Resposta"))))))()
}


## §1 Table 1 - Inventário de Saúde Mental exclusivo para Líderes

Líderes: 61 respostas, sendo 46 da FINX (T1 = 15, T2 = 16, T3 = 15) e 15 da Xforce (T1 = 6, T2 = 6, T3 = 3). As respostas não podem ser pareadas entre as ondas, porque não há identificação do respondente. As coletas foram em dez/2025–jan/2026 (T1), jun/2026 (T2) e ago/2026 (T3).

Os principais resultados indicam:
- FINX: a percepção de pressão constante por resultados aumentou de T2 para T3 (W = 27.5; p < 0.001). A concordância foi de 3/16 (19%) para 11/15 (73%). A redação do item no T2 é diferente.
- Xforce: a necessidade de intervir em conflitos diminuiu de T1 para T2 (W = 32.0; p = 0.023). A discordância foi de 1/6 para 5/6.
- Nos demais itens não houve diferença significativa entre as ondas (p ≥ 0.05).

In [2]:
tabela("lideres")

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,finx_T2→T3,finx_T3,xforce_T1,xforce_T1→T2,xforce_T2,xforce_T2→T3,xforce_T3
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
5. percebo pressao constante para entrega de resultados que exigem alto nivel de desempenho.,(1) discordo totalmente,2,W = 164.5; p = 0.073,6,W = 27.5; p < 0.001,0,1,W = 28.5; p = 0.096,3,W = 3.5; p = 0.179,0
,(2) discordo parcialmente,3,,4,,1,0,,1,,1
,(3) nem concordo nem discordo,4,,3,,3,1,,1,,0
,(4) concordo parcialmente,5,,3,,4,3,,1,,2
,(5) concordo totalmente,1,,0,,7,1,,0,,0
7. preciso intervir frequentemente em conflitos que desgastam minha rotina de trabalho.,(1) discordo totalmente,6,W = 106.5; p = 0.587,7,W = 93.0; p = 0.276,3,0,W = 32.0; p = 0.023,3,W = 6.5; p = 0.583,1
,(2) discordo parcialmente,6,,3,,3,1,,2,,1
,(3) nem concordo nem discordo,3,,3,,7,4,,1,,0
,(4) concordo parcialmente,0,,3,,1,1,,0,,1


## §2 Table 2 - Inventário de Saúde Emocional

Saúde Emocional: 216 respostas, sendo 172 da FINX (T1 = 86, T2 = 86) e 44 da Xforce (T1 = 25, T2 = 19). As respostas não podem ser pareadas entre as ondas, porque não há identificação do respondente. As coletas foram em dez/2025–jan/2026 (T1) e abr/2026 (T2).

Os principais resultados indicam:
- FINX: a liberdade para organizar as próprias atividades diminuiu de T1 para T2 (W = 4596.0; p < 0.001). "Muitas vezes" e "sempre" passaram de 91% para 73%.
- FINX: o encorajamento para usar criatividade e conhecimento diminuiu de T1 para T2 (W = 4342.0; p = 0.015). "Muitas vezes" e "sempre" passaram de 80% para 73%.
- Xforce: nenhum item teve diferença significativa entre T1 e T2 (p ≥ 0.05).

In [3]:
tabela("emocional")

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,xforce_T1,xforce_T1→T2,xforce_T2
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
34. tem liberdade para organizar suas atividades? (ex.: escolher a ordem das tarefas),(1) nunca,0,W = 4596.0; p < 0.001,1,0,W = 178.0; p = 0.202,0
,(2) raramente,3,,5,1,,0
,(3) as vezes,5,,17,5,,2
,(4) muitas vezes,40,,40,14,,10
,(5) sempre,38,,21,5,,6
35. e encorajado(a) a usar criatividade e conhecimento? (ex.: sugerir melhorias ou novas ideias),(1) nunca,1,W = 4342.0; p = 0.015,1,1,W = 243.0; p = 0.650,0
,(2) raramente,5,,6,1,,3
,(3) as vezes,11,,16,6,,4
,(4) muitas vezes,30,,40,11,,7


## §3 Table 3 - Inventário de Saúde Mental exclusivo para Líderes (3 categorias)

Líderes: 61 respostas, sendo 46 da FINX (T1 = 15, T2 = 16, T3 = 15) e 15 da Xforce (T1 = 6, T2 = 6, T3 = 3). As respostas não podem ser pareadas entre as ondas, porque não há identificação do respondente. As coletas foram em dez/2025–jan/2026 (T1), jun/2026 (T2) e ago/2026 (T3).

Os principais resultados indicam:
- FINX: a percepção de pressão constante por resultados aumentou de T2 para T3 (W = 27.5; p < 0.001). "Concordo" foi de 3/16 (19%) para 11/15 (73%). A redação do item no T2 é diferente.
- Xforce: a necessidade de intervir em conflitos diminuiu de T1 para T2 (W = 32.0; p = 0.023). "Discordo" foi de 1/6 para 5/6.
- Nos demais itens não houve diferença significativa entre as ondas (p ≥ 0.05).

(risco) = item de risco: maior concordância/frequência indica maior risco.

In [4]:
tabela("lideres", agrupar = TRUE)

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,finx_T2→T3,finx_T3,xforce_T1,xforce_T1→T2,xforce_T2,xforce_T2→T3,xforce_T3
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
5. percebo pressao constante para entrega de resultados que exigem alto nivel de desempenho. (risco),(1) discordo,5,W = 164.5; p = 0.073,10,W = 27.5; p < 0.001,1,1,W = 28.5; p = 0.096,4,W = 3.5; p = 0.179,1
,(2) neutro,4,,3,,3,1,,1,,0
,(3) concordo,6,,3,,11,4,,1,,2
7. preciso intervir frequentemente em conflitos que desgastam minha rotina de trabalho. (risco),(1) discordo,12,W = 106.5; p = 0.587,10,W = 93.0; p = 0.276,6,1,W = 32.0; p = 0.023,5,W = 6.5; p = 0.583,2
,(2) neutro,3,,3,,7,4,,1,,0
,(3) concordo,0,,3,,2,1,,0,,1
"4. trabalho alem do horario previsto devido as demandas da lideranca (ex.: responder mensagens fora do expediente, reunioes extras). (risco)",(1) discordo,8,W = 136.5; p = 0.515,8,W = 72.5; p = 0.057,5,3,W = 23.0; p = 0.418,4,W = 6.0; p = 0.477,2
,(2) neutro,1,,3,,1,1,,2,,0
,(3) concordo,6,,5,,9,2,,0,,1


## §4 Table 4 - Inventário de Saúde Emocional (3 categorias)

Saúde Emocional: 216 respostas, sendo 172 da FINX (T1 = 86, T2 = 86) e 44 da Xforce (T1 = 25, T2 = 19). As respostas não podem ser pareadas entre as ondas, porque não há identificação do respondente. As coletas foram em dez/2025–jan/2026 (T1) e abr/2026 (T2).

Os principais resultados indicam:
- FINX: a liberdade para organizar as próprias atividades diminuiu de T1 para T2 (W = 4596.0; p < 0.001). "Muitas vezes/sempre" passou de 78/86 (91%) para 61/84 (73%).
- FINX: o encorajamento para usar criatividade e conhecimento diminuiu de T1 para T2 (W = 4342.0; p = 0.015). "Muitas vezes/sempre" passou de 69/86 (80%) para 61/84 (73%).
- Xforce: nenhum item teve diferença significativa entre T1 e T2 (p ≥ 0.05).

(risco) = item de risco: maior concordância/frequência indica maior risco.

In [5]:
tabela("emocional", agrupar = TRUE)

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,xforce_T1,xforce_T1→T2,xforce_T2
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
34. tem liberdade para organizar suas atividades? (ex.: escolher a ordem das tarefas),(1) nunca/raramente,3,W = 4596.0; p < 0.001,6,1,W = 178.0; p = 0.202,0
,(2) as vezes,5,,17,5,,2
,(3) muitas vezes/sempre,78,,61,19,,16
35. e encorajado(a) a usar criatividade e conhecimento? (ex.: sugerir melhorias ou novas ideias),(1) nunca/raramente,6,W = 4342.0; p = 0.015,7,2,W = 243.0; p = 0.650,3
,(2) as vezes,11,,16,6,,4
,(3) muitas vezes/sempre,69,,61,17,,11
4. cobra demais de si mesmo(a)? (ex.: sentir que nunca fez o suficiente) (risco),(1) nunca/raramente,3,W = 4289.0; p = 0.054,5,2,W = 243.0; p = 0.898,0
,(2) as vezes,14,,20,4,,3
,(3) muitas vezes/sempre,69,,61,19,,16
